### Mount drive and load data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip /content/drive/MyDrive/AML-Project/market.zip -d /content/Market-Pytorch

Output streaming troncato alle ultime 5000 righe.
  inflating: /content/Market-Pytorch/Market/multi-query/0221/0221_c6s1_046526_00.jpg  
  inflating: /content/Market-Pytorch/Market/multi-query/0221/0221_c3s1_045951_00.jpg  
   creating: /content/Market-Pytorch/Market/multi-query/0013/
  inflating: /content/Market-Pytorch/Market/multi-query/0013/0013_c5s1_000476_00.jpg  
  inflating: /content/Market-Pytorch/Market/multi-query/0013/0013_c6s1_000451_00.jpg  
  inflating: /content/Market-Pytorch/Market/multi-query/0013/0013_c5s1_000426_00.jpg  
  inflating: /content/Market-Pytorch/Market/multi-query/0013/0013_c5s1_000551_00.jpg  
  inflating: /content/Market-Pytorch/Market/multi-query/0013/0013_c6s1_000476_00.jpg  
   creating: /content/Market-Pytorch/Market/multi-query/0477/
  inflating: /content/Market-Pytorch/Market/multi-query/0477/0477_c1s2_056796_00.jpg  
  inflating: /content/Market-Pytorch/Market/multi-query/0477/0477_c3s1_124733_00.jpg  
  inflating: /content/Market-Pytorch/Market

In [3]:
!unzip /content/drive/MyDrive/AML-Project/cuhk03-converted.zip -d /content/cuhk03-converted

Archive:  /content/drive/MyDrive/AML-Project/cuhk03-converted.zip
   creating: /content/cuhk03-converted/gallery/
   creating: /content/cuhk03-converted/gallery/0095/
  inflating: /content/cuhk03-converted/gallery/0095/0095_c2_09.png  
  inflating: /content/cuhk03-converted/gallery/0095/0095_c1_03.png  
  inflating: /content/cuhk03-converted/gallery/0095/0095_c1_02.png  
  inflating: /content/cuhk03-converted/gallery/0095/0095_c1_01.png  
  inflating: /content/cuhk03-converted/gallery/0095/0095_c2_10.png  
  inflating: /content/cuhk03-converted/gallery/0095/0095_c2_07.png  
  inflating: /content/cuhk03-converted/gallery/0095/0095_c2_06.png  
   creating: /content/cuhk03-converted/gallery/0061/
  inflating: /content/cuhk03-converted/gallery/0061/0061_c2_06.png  
  inflating: /content/cuhk03-converted/gallery/0061/0061_c2_10.png  
  inflating: /content/cuhk03-converted/gallery/0061/0061_c1_04.png  
  inflating: /content/cuhk03-converted/gallery/0061/0061_c1_01.png  
  inflating: /content

### Load Libraries

In [5]:
import torch
import torch.nn as nn
from torch.nn import init
import torch.optim as optim
from torchvision import models
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.models.vision_transformer import vit_b_16
from torchvision.models import ViT_B_16_Weights

import matplotlib.pyplot as plt

import cv2

import os
import shutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
import timm
import json
import numpy as np
from PIL import Image
import copy
import random
import itertools
import re


device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(12)
np.random.seed(12)
random.seed(12)

# Testing

In [6]:
def weights_init_kaiming(m):
    classname = m.__class__.__name__
    # print(classname)
    if classname.find('Conv') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
    elif classname.find('Linear') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_out')
        init.constant_(m.bias.data, 0.0)
    elif classname.find('BatchNorm1d') != -1:
        init.normal_(m.weight.data, 1.0, 0.02)
        init.constant_(m.bias.data, 0.0)

def weights_init_classifier(m):
    classname = m.__class__.__name__
    if classname.find('Linear') != -1:
        init.normal_(m.weight.data, std=0.001)
        init.constant_(m.bias.data, 0.0)

class ClassBlock(nn.Module):
    def __init__(self, input_dim, class_num, droprate, relu=False, bnorm=True, num_bottleneck=512, linear=True, return_f = False):
        super(ClassBlock, self).__init__()
        self.return_f = return_f
        add_block = []
        if linear:
            add_block += [nn.Linear(input_dim, num_bottleneck)]
        else:
            num_bottleneck = input_dim
        if bnorm:
            add_block += [nn.BatchNorm1d(num_bottleneck)]
        if relu:
            add_block += [nn.LeakyReLU(0.1)]
        if droprate>0:
            add_block += [nn.Dropout(p=droprate)]
        add_block = nn.Sequential(*add_block)
        add_block.apply(weights_init_kaiming)

        classifier = []
        classifier += [nn.Linear(num_bottleneck, class_num)]
        classifier = nn.Sequential(*classifier)
        classifier.apply(weights_init_classifier)

        self.add_block = add_block
        self.classifier = classifier
    def forward(self, x):
        x = self.add_block(x)
        if self.return_f:
            f = x
            x = self.classifier(x)
            return [x,f]
        else:
            x = self.classifier(x)
            return x

class LATransformer(nn.Module):
    def __init__(self, model, lmbd, print_verbose = False, test=False, pretraining=False):
        super(LATransformer, self).__init__()

        if print_verbose:
            self._print = print
        else:
            self._print = lambda *args, **kwargs: None
        self.class_num = 751
        self.part = 14 # We cut the pool5 to sqrt(N) parts
        self.num_blocks = 12
        self.model = model
        self.model.head.requires_grad_ = False
        self.cls_token = self.model.cls_token
        self.pos_embed = self.model.pos_embed
        self.avgpool = nn.AdaptiveAvgPool2d((self.part,768))
        self.dropout = nn.Dropout(p=0.5)
        self.lmbd = lmbd
        self.test = test
        self.pretraining = pretraining
        if not (self.test or self.pretraining):
          for i in range(self.part):
              name = 'classifier'+str(i)
              setattr(self, name, ClassBlock(768, self.class_num, droprate=0.5, relu=False, bnorm=True, num_bottleneck=256))

        if self.pretraining:
          self.fc = nn.Sequential(nn.Conv1d(14, 32, 3),
                                  nn.BatchNorm1d(32),
                                  nn.LeakyReLU(0.1),
                                  nn.Conv1d(32, 3, 3),
                                  nn.BatchNorm1d(3),
                                  nn.LeakyReLU(0.1),
                                  nn.Flatten(),
                                  nn.Linear(2292, 1024),
                                  nn.LeakyReLU(0.1),
                                  nn.Linear(1024,128))

          self.fc.apply(weights_init_kaiming)



    def forward(self,x):

        # Divide input image into patch embeddings and add position embeddings
        # cls token is a learnable parameter added to the start of sequence
        # It contains global info about the whole image
        # Used in classical Transformers like BERT and ViT to do classification with just itself
        # Here it is later combined to enrich local features with global features of the image
        self._print(f"x before pos embedding: {x.shape}")
        x = self.model.patch_embed(x)
        self._print(f"x after pos embedding: {x.shape}")
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        self._print(f"cls token: {cls_token.shape}")
        x = torch.cat((cls_token, x), dim=1)
        self._print(f"x with concatenation with cls token: {x.shape}")
        x = self.model.pos_drop(x + self.pos_embed)
        self._print(f"x after pos drop: {x.shape}")

        # Feed forward through transformer blocks
        for i in range(self.num_blocks):
            self._print(f"x before block {i}: {x.shape}")
            x = self.model.blocks[i](x)
        x = self.model.norm(x)
        self._print(f"x after blocks: {x.shape}")

        # extract the cls token
        cls_token_out = x[:, 0].unsqueeze(1)
        self._print(f"cls token out: {cls_token_out.shape}")

        # Average pool
        x = self.avgpool(x[:, 1:])
        self._print(f"x after avgpool: {x.shape}")

        if self.test:
          return x

        # Add global cls token to each local token
        for i in range(self.part):
            self._print(f"x before mul: {x.shape}")
            out = torch.mul(x[:, i, :], self.lmbd)
            x[:,i,:] = torch.div(torch.add(cls_token_out.squeeze(),out), 1+self.lmbd)

        if self.pretraining:
          x = x.reshape(x.size(0), 14, -1)
          x = self.fc(x)
          return x

        # Locally aware network
        part = {}
        predict = {}
        for i in range(self.part):
            part[i] = x[:,i,:]
            name = 'classifier'+str(i)
            c = getattr(self,name)
            predict[i] = c(part[i])
        return predict

In [7]:
class ProjectionHead(nn.Module):
    def __init__(self, embed_dim = 768):
        super(ProjectionHead, self).__init__()

        self.multihead_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)

        # Layer Normalization
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)

        # Feedforward with GELU Activation
        self.feedforward = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.GELU(),
            nn.Linear(1024, embed_dim)
        )

        # Adaptive Average Pooling
        self.gem = GeMPooling(output_size = 5)

        self.dropout_attn = nn.Dropout(p=0.1)
        self.dropout_ff = nn.Dropout(p=0.1)

    def forward(self, x):
        attn_output, _ = self.multihead_attn(x, x, x)  # [BATCH, 197, 768]
        attn_output = self.dropout_attn(attn_output)
        x = x + attn_output  # Residual Connection
        x = self.layer_norm1(x)  # Layer Norm

        # Feedforward Layer
        ff_output = self.feedforward(x)  # [BATCH, 197, 768]
        ff_output = self.dropout_ff(ff_output)
        x = x + ff_output  # Residual Connection
        x = self.layer_norm2(x)  # Layer Norm

        x = self.gem(x)  # [BATCH, 197, 5]

        # Permute back and average: [BATCH, 985]
        x = x.view(x.size(0), -1)

        return x

class ContrastiveTransformer(nn.Module):
    def __init__(self, freeze=True):
        super(ContrastiveTransformer, self).__init__()
        self.backbone = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        if freeze:
          for param in self.backbone.parameters():
            param.requires_grad = False
        self.projection_head = ProjectionHead()

    def forward(self,x):
        x = self.backbone._process_input(x)
        batch_class_token = self.backbone.class_token.expand(x.shape[0], -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = self.backbone.encoder(x)

        x = self.projection_head(x)
        return x

class GeMPooling(nn.Module):
    def __init__(self, p=3.0, output_size=1):
        super(GeMPooling, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.output_size = output_size

    def forward(self, x):
        return F.adaptive_avg_pool1d(x.clamp(min=1e-6).pow(self.p), self.output_size).pow(1.0 / self.p)

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.model1 = ContrastiveTransformer()
        self.model2 = self.model1

    def forward(self, x1, x2 = None):
        y1 = self.model1(x1)

        if x2 is None:
          return y1
        else:
          y2 = self.model2(x2)

        return y1, y2

In [8]:
def create_test_model(path):
  vit_base = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=751)
  vit_base= vit_base.to(device)
  if re.search("contrastive", path, re.IGNORECASE):
    model = SiameseNetwork()
  else:
    model = LATransformer(vit_base, lmbd=8, test=True).to(device)

  model.load_state_dict(torch.load(path), strict=False)
  model.eval()

  return model

In [9]:
def create_test_data(data_dir):
  batch_size = 8

  transform_query_list = transforms.Compose([
      transforms.Resize((224,224), interpolation=3),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
      ])

  transform_gallery_list = transforms.Compose([
      transforms.Resize(size=(224,224),interpolation=3), #Image.BICUBIC
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
      ])

  dataset_query = datasets.ImageFolder(os.path.join(data_dir, 'query'),transform=transform_query_list)
  dataset_gallery = datasets.ImageFolder(os.path.join(data_dir, 'gallery'),transform=transform_gallery_list)
  query_loader = DataLoader(dataset = dataset_query, batch_size=batch_size, shuffle=False)
  gallery_loader = DataLoader(dataset = dataset_gallery, batch_size=batch_size, shuffle=False)

  return dataset_query, dataset_gallery, query_loader, gallery_loader

In [10]:
def extract_feature(model,dataloader):

    features = torch.FloatTensor()
    count = 0
    idx = 0
    for data in tqdm(dataloader, desc="Extracting features"):
        img, label = data
        img, label = img.to(device), label.to(device)

        output = model(img)

        b, c, h, w = img.size()

        count += b
        features = torch.cat((features, output.detach().cpu()), 0)
        idx += 1
    return features

In [11]:
def get_labels(img_path):
    labels = []
    for path, v in img_path:
        filename = os.path.basename(path)
        label = filename[0:4]
        if label[0:2]=='-1':
            labels.append(-1)
        else:
            labels.append(int(label))
    return labels

def normalize(features):
    return features / features.norm(dim=-1, keepdim=True)

def compute_similarity(query_features, gallery_features):
    query_features = normalize(query_features)
    gallery_features = normalize(gallery_features)
    return torch.mm(query_features, gallery_features.t())

def evaluate_ranking(similarity_matrix, query_labels, gallery_labels, k_vals=[1, 5, 10]):
    num_queries = similarity_matrix.size(0)
    num_gallery = similarity_matrix.size(1)
    rank_k = {k: 0 for k in k_vals}
    mAP = 0.0

    gallery_labels = torch.tensor(gallery_labels, device=similarity_matrix.device)
    query_labels = torch.tensor(query_labels, device=similarity_matrix.device)

    for i in range(num_queries):
        sorted_indices = torch.argsort(similarity_matrix[i], descending=True)

        sorted_labels = gallery_labels[sorted_indices]
        relevant = (sorted_labels == query_labels[i]).cpu().numpy()

        similarity_scores = similarity_matrix[i][sorted_indices].cpu().numpy()
        ap = average_precision_score(relevant, similarity_scores)
        mAP += ap
        for k in k_vals:
            if relevant[:k].any():
                rank_k[k] += 1

    rank_k = {k: rank_k[k] / num_queries for k in k_vals}
    mAP = mAP / num_queries

    return rank_k, mAP

In [12]:
def test(model_path, data_dir="/content/Market-Pytorch/Market"):
  dataset_query, dataset_gallery, query_loader, gallery_loader = create_test_data(data_dir)
  model = create_test_model(model_path)
  query_features = extract_feature(model, query_loader)
  gallery_features = extract_feature(model, gallery_loader)

  query_features = query_features.view(query_features.size(0), -1)
  gallery_features = gallery_features.view(gallery_features.size(0), -1)

  similarity_matrix = compute_similarity(query_features, gallery_features)

  gallery_path = dataset_gallery.imgs
  query_path = dataset_query.imgs

  gallery_labels = get_labels(gallery_path)
  query_labels = get_labels(query_path)

  rank_k, mAP = evaluate_ranking(similarity_matrix, query_labels, gallery_labels)

  print("Rank1: {}, Rank5: {}, Rank10: {}, mAP: {}".format(rank_k[1], rank_k[5], rank_k[10], mAP))

  return rank_k[1], rank_k[5], rank_k[10], mAP

# Market-1501 Tests

Testing:
 - Model: LA Transformer
 - Pretrained: ImageNet
 - Fine Tuned: Market1501
 - Test Dataset: Market1501

In [ ]:
test(model_path = "/content/drive/MyDrive/AML-Project/LA-Transformers(Vanilla)/model_epoch_30.pth")

<ipython-input-8-30b0f8624a6c>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path), strict=False)


Extracting features:   0%|          | 0/289 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/2178 [00:00<?, ?it/s]

Rank1: 0.8964919878735383, Rank5: 0.9740147249891729, Rank10: 0.9818103074924209, mAP: 0.6923885321139353


(0.8964919878735383,
 0.9740147249891729,
 0.9818103074924209,
 0.6923885321139353)

Testing:
 - Model: LA Transformer
 - Pretrained: LUPerson
 - Fine Tuned: Market1501
 - Test Dataset: Market1501

In [ ]:
test(model_path = "/content/drive/MyDrive/AML-Project/LA-Transformers(Pretrained+FineTuned)/model_epoch_30.pth")

<ipython-input-8-30b0f8624a6c>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path), strict=False)


Extracting features:   0%|          | 0/289 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/2178 [00:00<?, ?it/s]

Rank1: 0.8925941966219142, Rank5: 0.9787786920744911, Rank10: 0.9844088349935037, mAP: 0.7004868185466738


(0.8925941966219142,
 0.9787786920744911,
 0.9844088349935037,
 0.7004868185466738)

Testing:
 - Model: Contrastive Transformer
 - Pretrained: ImageNet
 - Fine Tuned: Market1501
 - Test Dataset: Market1501

In [ ]:
test(model_path = "/content/drive/MyDrive/AML-Project/Constrastive_Transformers/model_epoch_100.pth")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

<ipython-input-8-458e3aab4cc1>:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path), strict=False)


Extracting features:   0%|          | 0/289 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/2178 [00:00<?, ?it/s]

Rank1: 0.49112169770463404, Rank5: 0.6691208315288003, Rank10: 0.7345171069727154, mAP: 0.10517259965037981


(0.49112169770463404,
 0.6691208315288003,
 0.7345171069727154,
 0.10517259965037981)

# CUHK-03 Tests

Testing:
 - Model: LA Transformer
 - Pretrained: ImageNet
 - Fine Tuned: Market1501
 - Test Dataset: CUHK03

In [ ]:
test(model_path="/content/drive/MyDrive/AML-Project/LA-Transformers(Vanilla)/model_epoch_30.pth", data_dir="/content/cuhk03-converted")

<ipython-input-8-30b0f8624a6c>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path), strict=False)


Extracting features:   0%|          | 0/23 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/98 [00:00<?, ?it/s]

Rank1: 0.8342541436464088, Rank5: 0.9281767955801105, Rank10: 0.9613259668508287, mAP: 0.4857381479557377


(0.8342541436464088,
 0.9281767955801105,
 0.9613259668508287,
 0.4857381479557377)

Testing:
 - Model: LA Transformer
 - Pretrained: LUPerson
 - Fine Tuned: Market1501
 - Test Dataset: CUHK03

In [ ]:
test(model_path="/content/drive/MyDrive/AML-Project/LA-Transformers(Pretrained+FineTuned)/model_epoch_30.pth", data_dir="/content/cuhk03-converted")

<ipython-input-8-30b0f8624a6c>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path), strict=False)


Extracting features:   0%|          | 0/23 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/98 [00:00<?, ?it/s]

Rank1: 0.8453038674033149, Rank5: 0.9502762430939227, Rank10: 0.9502762430939227, mAP: 0.48230534567605726


(0.8453038674033149,
 0.9502762430939227,
 0.9502762430939227,
 0.48230534567605726)

Testing:
 - Model: Contrastive Transformer
 - Pretrained: ImageNet
 - Fine Tuned: Market1501
 - Test Dataset: CHUK03

In [13]:
test(model_path = "/content/drive/MyDrive/AML-Project/Constrastive_Transformers/model_epoch_100.pth", data_dir="/content/cuhk03-converted")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

<ipython-input-8-458e3aab4cc1>:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path), strict=False)


Extracting features:   0%|          | 0/23 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/98 [00:00<?, ?it/s]

Rank1: 0.7292817679558011, Rank5: 0.8950276243093923, Rank10: 0.9171270718232044, mAP: 0.27857508662212355


(0.7292817679558011,
 0.8950276243093923,
 0.9171270718232044,
 0.27857508662212355)